# Vendor Performance & Cost Optimization Analysis

This notebook reframes the raw transaction data as an executive decision tool. The focus is not just on descriptive statistics, but on the operational levers that matter most in vendor management: spend concentration, inventory risk, category profitability, and lead-time efficiency.


In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

sns.set_theme(style="whitegrid", palette="crest")
plt.rcParams["figure.figsize"] = (12, 6)
plt.rcParams["axes.titlesize"] = 14
plt.rcParams["axes.labelsize"] = 11


In [ ]:
# Load curated summary tables created from the large raw transaction files.
vendor_summary = pd.read_csv("../data/vendor_performance_summary.csv")
vendor_brand = pd.read_csv("../data/vendor_brand_performance.csv")
category_summary = pd.read_csv("../data/category_performance_summary.csv")

vendor_summary.head()


## Portfolio Snapshot

These KPIs give a management-level view before drilling into vendor or category detail.


In [ ]:
kpi_snapshot = pd.DataFrame(
    {
        "Metric": [
            "Vendor Count",
            "Total Revenue",
            "Total Purchase Cost",
            "Adjusted Profit",
            "Gross Margin %",
            "Inventory at Risk",
        ],
        "Value": [
            vendor_summary["VendorNumber"].nunique(),
            vendor_summary["Revenue"].sum(),
            vendor_summary["PurchaseCost"].sum(),
            vendor_summary["AdjustedProfit"].sum(),
            (vendor_summary["AdjustedProfit"].sum() / vendor_summary["Revenue"].sum()) * 100,
            vendor_summary["InventoryAtRisk"].sum(),
        ],
    }
)
kpi_snapshot


## EDA Insight 1: Spend Concentration Risk

Concentration matters because a portfolio that depends too heavily on a small vendor group is exposed to pricing power, supply disruption, and weaker negotiation leverage.


In [ ]:
top_spend = vendor_summary.nlargest(10, "PurchaseCost").copy()
top_spend["label"] = top_spend["VendorName"].str.slice(0, 28)

ax = sns.barplot(data=top_spend, x="PurchaseCost", y="label")
ax.set_title("Top 10 Vendors by Procurement Spend")
ax.set_xlabel("Purchase Cost")
ax.set_ylabel("Vendor")
plt.show()

top_10_spend_share = top_spend["PurchaseCost"].sum() / vendor_summary["PurchaseCost"].sum() * 100
top_10_spend_share


## EDA Insight 2: Inventory Imbalance and Cash Exposure

The next view isolates suppliers where purchase volumes are running ahead of demand, tying up working capital in slow-moving stock.


In [ ]:
inventory_risk = vendor_summary.nlargest(12, "InventoryAtRisk").copy()
inventory_risk["label"] = inventory_risk["VendorName"].str.slice(0, 28)

ax = sns.barplot(data=inventory_risk, x="InventoryAtRisk", y="label", color="#C05A2B")
ax.set_title("Vendors Creating the Largest Inventory Risk")
ax.set_xlabel("Inventory at Risk")
ax.set_ylabel("Vendor")
plt.show()


## EDA Insight 3: Lead Time Efficiency vs Profitability

This adds an operational lens. Vendors with long lead times and weak margins usually deserve immediate review because they combine service risk with poor commercial return.


In [ ]:
plt.figure(figsize=(12, 7))
sns.scatterplot(
    data=vendor_summary,
    x="AvgLeadDays",
    y="AdjustedMarginPct",
    size="Revenue",
    hue="PerformanceTier",
    sizes=(40, 500),
    alpha=0.8,
)
plt.title("Lead Time vs Adjusted Margin")
plt.xlabel("Average Lead Days")
plt.ylabel("Adjusted Margin %")
plt.legend(bbox_to_anchor=(1.02, 1), loc="upper left")
plt.show()


## EDA Insight 4: Category Profit Pools

Category profitability helps distinguish where procurement effort should be focused. Some categories drive revenue, while others do a better job of converting spend into profit.


In [ ]:
category_view = category_summary.sort_values("AdjustedProfit", ascending=False).copy()

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
sns.barplot(data=category_view, x="Classification", y="Revenue", ax=axes[0], color="#4472C4")
sns.barplot(data=category_view, x="Classification", y="AdjustedProfit", ax=axes[1], color="#70AD47")

axes[0].set_title("Revenue by Category")
axes[0].set_xlabel("Classification")
axes[0].set_ylabel("Revenue")
axes[1].set_title("Adjusted Profit by Category")
axes[1].set_xlabel("Classification")
axes[1].set_ylabel("Adjusted Profit")
plt.tight_layout()
plt.show()


## What This Means for the Business

The curated summary makes it easier to separate three vendor stories:

1. High-impact vendors that deserve partnership and volume planning.
2. Cost-heavy vendors that need margin negotiation or mix correction.
3. Long-lead, low-return vendors that deserve a watchlist or exit plan.
